# 19. Переобучение Hierarchical Span NER на global_v1

Обучение выполняется во временном `/content`; в Drive переносится только лучший checkpoint и компактные метрики.

In [ ]:
from pathlib import Path
import os, runpy
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
if not PROJECT_DIR.exists(): PROJECT_DIR = Path.cwd()
os.environ['HF_HOME'] = '/content/huggingface_cache'
runpy.run_path(str(PROJECT_DIR / 'colab_bootstrap.py'))['bootstrap_project'](PROJECT_DIR)


In [ ]:
from datetime import datetime
from rurebus_ie.configuration import load_experiment_bundle
from rurebus_ie.training import train_hierarchical_span_ner_experiment, persist_best_run

CONFIG = PROJECT_DIR / 'configs/experiments/hierarchical_span_ner_global_v1.yaml'
bundle = load_experiment_bundle(CONFIG, project_root=PROJECT_DIR)
PERSISTENT_RUN = Path(bundle['experiment']['output_dir'])
LOCAL_RUN = Path('/content/rurebus_runs') / f"hierarchical_span_global_v1_{datetime.now():%Y%m%d_%H%M%S}"
print('Временный run:', LOCAL_RUN)
print('Итоговый best checkpoint:', PERSISTENT_RUN)
summary = train_hierarchical_span_ner_experiment(CONFIG, project_root=PROJECT_DIR, output_dir_override=LOCAL_RUN)
storage = persist_best_run(LOCAL_RUN, PERSISTENT_RUN)
print(f"Best epoch: {summary.best_epoch}; validation micro-F1: {summary.best_validation_f1:.6f}")
print(f"Сохранено в Drive: {storage['total_bytes'] / 2**30:.2f} GiB (только best).")


In [ ]:
import gc
try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()
print('GPU cache очищен; временный run останется только до завершения Colab runtime.')
